# Этап 4. Baseline-модель и первая accuracy

Colab-ноутбук для первого прототипа классификации дефектов поверхности стали.

Что делает ноутбук:
- загружает `train / val / test` манифесты из этапа 3;
- подготавливает датасеты и `DataLoader`;
- обучает baseline-модель `ResNet18`;
- считает первую `accuracy` на тесте;
- строит графики обучения;
- сохраняет веса, историю обучения и метрики.

## Как использовать

1. Положите проект в Google Drive.
2. Убедитесь, что есть папки `DB` и `stage3_outputs`.
3. Включите GPU в Colab: `Runtime -> Change runtime type -> GPU`.
4. Запускайте ноутбук сверху вниз.
5. Артефакты сохранятся в `stage4_outputs`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/MyProject1')
DATASET_ROOT = PROJECT_ROOT / 'DB'
MANIFEST_DIR = PROJECT_ROOT / 'stage3_outputs'
OUTPUT_DIR = PROJECT_ROOT / 'stage4_outputs'
MODEL_DIR = OUTPUT_DIR / 'models'
PLOT_DIR = OUTPUT_DIR / 'plots'
REPORT_DIR = OUTPUT_DIR / 'reports'

for directory in [OUTPUT_DIR, MODEL_DIR, PLOT_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = MANIFEST_DIR / 'train_manifest.csv'
VAL_CSV = MANIFEST_DIR / 'val_manifest.csv'
TEST_CSV = MANIFEST_DIR / 'test_manifest.csv'

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
RANDOM_SEED = 42
MODEL_NAME = 'resnet18'
USE_PRETRAINED = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('DEVICE     =', DEVICE)
print('TRAIN_CSV  =', TRAIN_CSV)
print('VAL_CSV    =', VAL_CSV)
print('TEST_CSV   =', TEST_CSV)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def rebuild_image_path(image_path: str, source_folder: str, dataset_root: Path) -> str:
    image_name = Path(image_path).name
    return str(dataset_root / 'images' / 'images' / source_folder / image_name)


def load_manifest(csv_path: Path, dataset_root: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df['resolved_image_path'] = df.apply(
        lambda row: rebuild_image_path(row['image_path'], row['source_folder'], dataset_root),
        axis=1,
    )
    return df


def build_target_mapping(train_df: pd.DataFrame) -> tuple[dict[int, int], dict[int, str], dict[int, int]]:
    classes = (
        train_df[['class_id', 'class_name']]
        .drop_duplicates()
        .sort_values('class_id')
        .reset_index(drop=True)
    )

    class_id_to_target = {int(row.class_id): idx for idx, row in classes.iterrows()}
    target_to_class_name = {idx: row.class_name for idx, row in classes.iterrows()}
    target_to_class_id = {idx: int(row.class_id) for idx, row in classes.iterrows()}
    return class_id_to_target, target_to_class_name, target_to_class_id


def prepare_targets(df: pd.DataFrame, class_id_to_target: dict[int, int]) -> pd.DataFrame:
    out_df = df.copy()
    out_df['target'] = out_df['class_id'].map(class_id_to_target)
    return out_df


def validate_manifest_paths(df: pd.DataFrame, name: str) -> None:
    missing_paths = [path for path in df['resolved_image_path'] if not Path(path).exists()]
    print(f'{name}: {len(df)} записей')
    print(f'{name}: отсутствующих файлов = {len(missing_paths)}')
    assert len(missing_paths) == 0, f'В {name} есть пути к отсутствующим изображениям.'


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
class SteelDefectDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['resolved_image_path']).convert('RGB')
        target = int(row['target'])

        if self.transform is not None:
            image = self.transform(image)

        return image, target


def build_transforms(image_size: int):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return train_transform, eval_transform


def create_dataloaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    image_size: int,
    batch_size: int,
    num_workers: int,
):
    train_transform, eval_transform = build_transforms(image_size)

    train_dataset = SteelDefectDataset(train_df, transform=train_transform)
    val_dataset = SteelDefectDataset(val_df, transform=eval_transform)
    test_dataset = SteelDefectDataset(test_df, transform=eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader

In [ ]:
def get_model(model_name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    if model_name != 'resnet18':
        raise ValueError('Для baseline-ноутбука поддерживается только model_name="resnet18".')

    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_targets = []
    all_preds = []

    for images, targets in tqdm(loader, desc='Train', leave=False):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        all_targets.extend(targets.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc


def evaluate_model(model, loader, criterion, device, desc='Eval'):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc=desc, leave=False):
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            all_targets.extend(targets.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc, all_targets, all_preds


def fit_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs):
    history = []
    best_val_acc = -1.0
    best_state_dict = None

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate_model(model, val_loader, criterion, device, desc='Val')

        epoch_time = time.time() - epoch_start
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_accuracy': train_acc,
            'val_loss': val_loss,
            'val_accuracy': val_acc,
            'epoch_time_sec': epoch_time,
        })

        print(
            f'Epoch {epoch:02d}/{num_epochs} | '
            f'train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | '
            f'val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | '
            f'time={epoch_time:.1f}s'
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = {key: value.cpu().clone() for key, value in model.state_dict().items()}

    history_df = pd.DataFrame(history)
    model.load_state_dict(best_state_dict)
    return model, history_df, best_val_acc

In [ ]:
def plot_history(history_df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='train_accuracy')
    axes[1].plot(history_df['epoch'], history_df['val_accuracy'], label='val_accuracy')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    return fig


def plot_confusion_matrix_figure(y_true, y_pred, target_to_class_name):
    labels = sorted(target_to_class_name.keys())
    class_names = [target_to_class_name[idx] for idx in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    return fig


def save_artifacts(
    model,
    history_df,
    test_metrics,
    target_to_class_name,
    target_to_class_id,
    output_dir,
    model_name,
):
    model_path = output_dir / 'models' / f'{model_name}_best_model.pth'
    history_path = output_dir / 'reports' / 'history.csv'
    metrics_path = output_dir / 'reports' / 'test_metrics.json'
    class_mapping_path = output_dir / 'reports' / 'class_mapping.json'

    torch.save(model.state_dict(), model_path)
    history_df.to_csv(history_path, index=False)

    serializable_mapping = {
        str(target): {
            'class_id': int(target_to_class_id[target]),
            'class_name': target_to_class_name[target],
        }
        for target in target_to_class_name
    }

    with open(metrics_path, 'w', encoding='utf-8') as fp:
        json.dump(test_metrics, fp, ensure_ascii=False, indent=2)

    with open(class_mapping_path, 'w', encoding='utf-8') as fp:
        json.dump(serializable_mapping, fp, ensure_ascii=False, indent=2)

    return {
        'model_path': str(model_path),
        'history_path': str(history_path),
        'metrics_path': str(metrics_path),
        'class_mapping_path': str(class_mapping_path),
    }

In [ ]:
def run_training_pipeline(
    train_csv: Path,
    val_csv: Path,
    test_csv: Path,
    dataset_root: Path,
    output_dir: Path,
    image_size: int,
    batch_size: int,
    num_workers: int,
    num_epochs: int,
    learning_rate: float,
    weight_decay: float,
    model_name: str,
    use_pretrained: bool,
    random_seed: int,
    device: str,
):
    set_seed(random_seed)

    train_df = load_manifest(train_csv, dataset_root)
    val_df = load_manifest(val_csv, dataset_root)
    test_df = load_manifest(test_csv, dataset_root)

    validate_manifest_paths(train_df, 'train')
    validate_manifest_paths(val_df, 'val')
    validate_manifest_paths(test_df, 'test')

    class_id_to_target, target_to_class_name, target_to_class_id = build_target_mapping(train_df)
    train_df = prepare_targets(train_df, class_id_to_target)
    val_df = prepare_targets(val_df, class_id_to_target)
    test_df = prepare_targets(test_df, class_id_to_target)

    train_loader, val_loader, test_loader = create_dataloaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        image_size=image_size,
        batch_size=batch_size,
        num_workers=num_workers,
    )

    model = get_model(model_name=model_name, num_classes=len(class_id_to_target), pretrained=use_pretrained)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    print('Параметров модели:', count_parameters(model))
    model, history_df, best_val_acc = fit_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
    )

    test_loss, test_acc, y_true, y_pred = evaluate_model(model, test_loader, criterion, device, desc='Test')
    report = classification_report(
        y_true,
        y_pred,
        labels=sorted(target_to_class_name.keys()),
        target_names=[target_to_class_name[idx] for idx in sorted(target_to_class_name.keys())],
        output_dict=True,
        zero_division=0,
    )

    history_fig = plot_history(history_df)
    history_fig_path = output_dir / 'plots' / 'loss_accuracy_curves.png'
    history_fig.savefig(history_fig_path, dpi=150, bbox_inches='tight')
    plt.show()

    cm_fig = plot_confusion_matrix_figure(y_true, y_pred, target_to_class_name)
    cm_fig_path = output_dir / 'plots' / 'confusion_matrix.png'
    cm_fig.savefig(cm_fig_path, dpi=150, bbox_inches='tight')
    plt.show()

    test_metrics = {
        'model_name': model_name,
        'best_val_accuracy': float(best_val_acc),
        'test_loss': float(test_loss),
        'test_accuracy': float(test_acc),
        'num_parameters': int(count_parameters(model)),
        'classification_report': report,
        'history_plot_path': str(history_fig_path),
        'confusion_matrix_path': str(cm_fig_path),
    }

    saved_paths = save_artifacts(
        model=model,
        history_df=history_df,
        test_metrics=test_metrics,
        target_to_class_name=target_to_class_name,
        target_to_class_id=target_to_class_id,
        output_dir=output_dir,
        model_name=model_name,
    )

    summary = {
        'train_count': int(len(train_df)),
        'val_count': int(len(val_df)),
        'test_count': int(len(test_df)),
        'num_classes': int(len(class_id_to_target)),
        'best_val_accuracy': float(best_val_acc),
        'test_accuracy': float(test_acc),
        'saved_paths': saved_paths,
    }

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary, history_df

In [ ]:
summary, history_df = run_training_pipeline(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    test_csv=TEST_CSV,
    dataset_root=DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    model_name=MODEL_NAME,
    use_pretrained=USE_PRETRAINED,
    random_seed=RANDOM_SEED,
    device=DEVICE,
)

history_df.tail()

## Что использовать в отчёте этапа 4

Из этого ноутбука в отчёт можно взять:
- описание задачи и постановки классификации;
- параметры обучения (`IMAGE_SIZE`, `BATCH_SIZE`, `NUM_EPOCHS`, `LEARNING_RATE`);
- архитектуру сети (`ResNet18`);
- графики `loss / accuracy`;
- итоговую `accuracy` на тестовой выборке;
- вывод о пригодности baseline-модели;
- план улучшений на 5 этап.